# Chapter 4: Yield Curves, Zero Rates, and Forward Rates

This notebook accompanies the chapter in *Python & AI for Rates, Bonds, and Credit*. It focuses on reproducible inspection: run the chapter script, inspect the data snapshot, and connect the numerical result back to the fixed-income interpretation.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT = ROOT / "code" / "ch04_yield_curves.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Use the script output as controlled evidence. Check the units, sign convention, source dataset, and the economic direction of the result before using it in a written explanation or GenAI-assisted summary.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch04_*.csv"))
summary = pd.DataFrame({
    "file": [p.name for p in chapter_files],
    "rows": [pd.read_csv(p).shape[0] for p in chapter_files],
    "columns": [pd.read_csv(p).shape[1] for p in chapter_files],
})
summary

## Chapter-specific checks

Compare a synthetic zero curve with the latest official Treasury snapshot, then inspect the forward-rate implication of log-discount interpolation.

In [ ]:
import numpy as np

zero = pd.read_csv(DATA / "ch04_zero_curve.csv")
zero["discount"] = np.exp(-zero.zero_rate * zero.maturity)
zero["forward"] = -np.log(zero.discount).diff() / zero.maturity.diff()
official = pd.read_csv(DATA / "ch04_us_treasury_par_yield_curve_official.csv")
latest = official.tail(1).T.rename(columns={official.index[-1]: "latest"})
zero.round(6), latest.head(8)